# Simulated Data: K-Means Clustering over 30 Runs

For each simulation scenario (`balanced`, `mild_imbalanced`, `strong_imbalanced`) and each
of 30 repetitions, run k-means on:
- **uncorrected** matrix (union of all lab `before/` matrices — identical across runs)
- **central corrected** matrix (`after/runs/{i}_R_corrected.tsv`)
- **federated corrected** matrix (`after/runs/{i}_FedSim_corrected.tsv`)

Targets: condition (k=2) and batch/lab (k=3).  
Outputs ARI per run, method, and target to `outputs/{scenario}_ari_results.tsv`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    load_feature_matrix,
    load_before_matrix_from_dir,
    filter_matrix_to_available,
    drop_rows_with_any_na,
    run_central_kmeans,
    align_predictions_to_truth,
    calculate_metrics,
)

In [ ]:
# -------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------
SCENARIOS = ["balanced", "mild_imbalanced", "strong_imbalanced"]
N_RUNS = 30
SEED = 11

K_CONDITION = 2   # two conditions (A, B)
K_BATCH = 3       # three labs (lab1, lab2, lab3)

DATA_ROOT = REPO_ROOT / "evaluation_data" / "simulated"
OUTPUT_ROOT = NOTEBOOK_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# -------------------------------------------------------------------------
# Main loop: iterate over scenarios and runs
# -------------------------------------------------------------------------

all_results = []

for scenario in SCENARIOS:
    scenario_dir = DATA_ROOT / scenario
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario}")
    print(f"{'='*60}")

    # Load metadata (same for all runs within scenario)
    metadata_path = scenario_dir / "all_metadata.tsv"
    metadata = pd.read_csv(metadata_path, sep="\t")
    metadata.columns = [c.strip().strip('"') for c in metadata.columns]
    metadata["file"] = metadata["file"].str.strip().str.strip('"')
    metadata["condition"] = metadata["condition"].str.strip().str.strip('"')
    metadata["lab"] = metadata["lab"].str.strip().str.strip('"')
    print(f"Metadata: {len(metadata)} samples, conditions={sorted(metadata['condition'].unique())}, labs={sorted(metadata['lab'].unique())}")

    # Load and prepare uncorrected matrix (shared across all runs)
    before_raw = load_before_matrix_from_dir(scenario_dir / "before")
    before_mat = filter_matrix_to_available(before_raw, metadata, f"{scenario}/before")
    before_meta = metadata[metadata["file"].isin(before_mat.columns)].copy()
    print(f"Before matrix: {before_mat.shape[0]} features × {before_mat.shape[1]} samples")

    for run_i in range(1, N_RUNS + 1):
        runs_dir = scenario_dir / "after" / "runs"
        central_path = runs_dir / f"{run_i}_R_corrected.tsv"
        fed_path = runs_dir / f"{run_i}_FedSim_corrected.tsv"

        if not central_path.exists():
            print(f"  [run {run_i}] Missing central corrected: {central_path} — skipping")
            continue
        if not fed_path.exists():
            print(f"  [run {run_i}] Missing federated corrected: {fed_path} — skipping")
            continue

        central_raw = load_feature_matrix(central_path)
        fed_raw = load_feature_matrix(fed_path)

        central_mat = filter_matrix_to_available(central_raw, metadata, f"{scenario}/run{run_i}/central")
        fed_mat = filter_matrix_to_available(fed_raw, metadata, f"{scenario}/run{run_i}/fed")

        # Samples present in all three matrices for a fair comparison
        common_samples = (
            set(before_mat.columns)
            & set(central_mat.columns)
            & set(fed_mat.columns)
        )
        run_meta = metadata[metadata["file"].isin(common_samples)].copy()
        if run_meta.empty:
            print(f"  [run {run_i}] No common samples — skipping")
            continue

        before_run = before_mat.loc[:, run_meta["file"]]
        central_run = central_mat.loc[:, run_meta["file"]]
        fed_run = fed_mat.loc[:, run_meta["file"]]

        matrices = {
            "uncorrected": before_run,
            "central": central_run,
            "federated": fed_run,
        }

        for method, mat in matrices.items():
            clusters = run_central_kmeans(mat, [K_CONDITION, K_BATCH], SEED)

            for target, k, truth_col in [
                ("condition", K_CONDITION, "condition"),
                ("batch", K_BATCH, "lab"),
            ]:
                predicted = clusters[k].rename(run_meta["file"].values)
                truth = run_meta[truth_col].values
                truth_series = pd.Series(truth, index=run_meta["file"].values)

                aligned = align_predictions_to_truth(predicted, truth_series)
                metrics = calculate_metrics(truth_series, aligned)

                all_results.append({
                    "scenario": scenario,
                    "run": run_i,
                    "method": method,
                    "target": target,
                    "ARI": metrics["ARI"],
                    "N": metrics["N"],
                })

        if run_i % 10 == 0:
            print(f"  Done runs 1–{run_i}")

print("\nAll scenarios complete.")

In [ ]:
# -------------------------------------------------------------------------
# Save results
# -------------------------------------------------------------------------

results_df = pd.DataFrame(all_results)

for scenario in SCENARIOS:
    subset = results_df[results_df["scenario"] == scenario]
    out_path = OUTPUT_ROOT / f"{scenario}_ari_results.tsv"
    subset.to_csv(out_path, sep="\t", index=False)
    print(f"Saved: {out_path}  ({len(subset)} rows)")

display(results_df.groupby(["scenario", "method", "target"])["ARI"].describe().round(3))